# Notebook 03 — Causal Feature Identification
**Owners: Person A (ablation) · Person B (CaFE, layers) · Person C (patch/CLS comparison)** — Week 2

**Goal:** Identify which SAE features causally drive flamingo vs. spoonbill. These become the circuit nodes.

**Key expected finding:** CaFE agreement rate well below 0.5.

**Rule:** Logic in `src/causal.py`.

In [ ]:
import sys; sys.path.insert(0, '..')
from src.config import get_config
cfg = get_config()
CLASS_A = 130  # flamingo
CLASS_B = 129  # spoonbill

## 1. Load behavior dataset
200 flamingo + 200 spoonbill, all correctly classified.

In [ ]:
from src.cache import load_layer, get_class_indices, load_image_index
from src.model import get_model

# TODO
# model = get_model()
# flamingo_idx = get_class_indices('flamingo')[:cfg.data.behavior_n_images]
# spoonbill_idx = get_class_indices('spoonbill')[:cfg.data.behavior_n_images]
# behavior_idx = flamingo_idx + spoonbill_idx
# activations_dict = {layer: load_layer(layer, indices=behavior_idx)
#                    for layer in cfg.sae.target_layers}

## 2. Baseline logit diff — Person A
Implement `causal.compute_logit_diff()`.

Test: positive for flamingo images, negative for spoonbill.

In [ ]:
from src.causal import compute_logit_diff

# TODO: compute baseline
# (needs pixel values, not activations — load images for behavior subset)
# baseline_diff = compute_logit_diff(model, behavior_pixels, CLASS_A, CLASS_B)
# print(f'Baseline logit diff: {baseline_diff.mean():.3f}')

## 3. Feature ablation importance — Person A
Implement `causal.ablation_importance()` for layer 11.

Test: top-ranked feature causes larger change than random.

In [ ]:
from src.causal import ablation_importance, get_top_causal_features

# TODO
# importance_11 = ablation_importance(
#     layer=11, activations=activations_dict[11],
#     model=model, class_a_idx=CLASS_A, class_b_idx=CLASS_B)
# top_features_11 = get_top_causal_features(importance_11)
# print(f'Top features layer 11: {len(top_features_11)}')

## 4. Multi-layer extension — Person B
Implement `causal.ablation_importance_multilayer()`. Reuse Person A's function.

In [ ]:
from src.causal import ablation_importance_multilayer
from src.visualise import plot_ablation_importance
import matplotlib.pyplot as plt

# TODO
# importance_all = ablation_importance_multilayer(
#     layers=cfg.sae.target_layers, activations_dict=activations_dict,
#     model=model, class_a_idx=CLASS_A, class_b_idx=CLASS_B)
# top_features_per_layer = {
#     layer: get_top_causal_features(importance_all[layer])
#     for layer in cfg.sae.target_layers}
# fig = plot_ablation_importance(importance_all[11], layer=11)
# plt.show()

## 5. CaFE sanity check — Person B
Implement `causal.cafe_sanity_check()` and `causal.cafe_agreement_rate()`.

**Key expected finding:** Agreement rate < 0.5 on DINOv2 (replicates Han et al. 2025 on a new model).

In [ ]:
from src.causal import cafe_sanity_check, cafe_agreement_rate
from src.visualise import plot_cafe_comparison
from src.cache import load_image_index

# TODO
# image_paths = load_image_index()['paths'].tolist()
# sample = top_features_11[:20]
# agreement = cafe_agreement_rate(11, sample, activations_dict[11], image_paths)
# print(f'CaFE agreement rate: {agreement:.2f}  (expected < 0.5)')
# cafe_results = [cafe_sanity_check(11, f, activations_dict[11], image_paths) for f in sample[:6]]
# fig = plot_cafe_comparison(cafe_results)
# fig.savefig('../report/figures/cafe_comparison.png', dpi=300, bbox_inches='tight')

## 6. Patch-token vs CLS comparison — Person C
Use `features.get_top_patches()` with `token_type='cls'` vs `token_type='patch'`.
Document: do the same features rank highly in both cases?

In [ ]:
from src.features import get_top_patches

# TODO: compare patch vs CLS importance for top 5 features
# Document findings in report/notes/person_C.md

## End-of-Week-2 checklist
- [ ] Baseline logit diff correct direction for both classes
- [ ] `ablation_importance()` shape `(d_sae,)`; ranking validated
- [ ] Top causal features identified for all 3 layers
- [ ] CaFE agreement rate < 0.5 ← key finding
- [ ] `plot_ablation_importance()` saved
- [ ] `plot_cafe_comparison()` saved to `report/figures/`